# HD 209458: transit timing demonstration

Demonstrate light-curve preparation, individual transit fitting, and timing residuals on another target. Review parameter choices and rejected windows before interpreting the fits. Timing trends require uncertainty analysis and an appropriate time baseline.

## Before running
Run from the repository root after installing `requirements.txt`. Archive access requires internet. Inspect the Lightkurve search table and explicitly choose the intended sector/product: numeric result indices are not stable archive identifiers. Saved execution outputs are intentionally absent.

## Review the results
Compare raw and cleaned flux; inspect masked regions and model residuals. Record the selected data product, assumptions, and uncertainties before using a result in scientific work.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
from mandelagol import occultquad
import lmfit as lmfit
from scipy.stats import chi2
plt.rcParams['font.size'] = 18
plt.rcParams['figure.figsize'] = [20,10]

In [ ]:
def tlc(params, t):

    tc = params['tc']
    period = params['period']
    u = params['u']
    k = params['k']
    b = params['b']
    a = params['a']
    flux_oot = params['flux_oot']
    dflux_dt = params['dflux_dt']

    '''
    
    Computes loss of light due to a transiting planet using formulas from Mandel & Agol (2002)
    and assuming linear limb-darkening law.

    Inputs:
     t0 = midpoint of transit
      k = (radius of planet) / (radius of star)
      a = (radius of orbit) / (radius of star)
      b = transit impact parameter = a*cos(incl), should be <= 1
      u = limb darkening parameter (0.0 for uniform intensity, 0.6 for Sun-like)
      flux_oot = value of flux outside of transits
      dflux_dt = fractional rate of change of flux_oot per day
      
    Outputs:
      calculated flux, with flux=1 outside of transit.
      
    '''
    
    phi = 2.*np.pi*(t-tc)/period  # the phase of the orbit
    x = a*np.sin(phi)             # the x-coordinate with reference to Fig. 2 of Winn (2010)
    y = b*np.cos(phi)             # the y-coordinate with reference to Fig. 2 of Winn (2010)
    z = np.sqrt(a*a-b*b)*np.cos(phi)
    s = np.sqrt(x*x + y*y)        # the distance between planet and star on the sky
    flux_calc, flux_no_limb_darkening = occultquad(s,u,0.0,k) # calculates the loss of light
    flux_calc[z<0.0] = 1.0        # dip occurs only when planet is in front of star (not behind)
    
    flux_calc_with_slope = flux_calc * flux_oot * (1.0 + dflux_dt * (t-tc)) 
    
    return(flux_calc_with_slope)

def tlc_residuals(params, t, f, f_unc):
    res = (f - tlc(params, t))/f_unc
    return(res)

def fold_time(t, tc_ref, period_ref):
    
    '''
    
    Converts time into "folded time" in the range between -period/2 and +period/2
    as well as an orbit number, assigning tc_ref = orbit zero
     
    '''  
    
    orbit_number, t_fold = np.divmod( t - (tc_ref - 0.5*period_ref), period_ref )
    orbit_number = orbit_number.astype(int)
    
    t_fold = t_fold - 0.5*period_ref
 
    return(t_fold, orbit_number)

def chisqr_pval(chisqr, ndof):
    
    '''
    
    inputs are chi-squared of best-fit model and number of degrees of freedom

    output is the probability that such a large chi-squared would be observed
    if the model is correct and the input data have independent Gaussian uncertainties
    
    '''
    
    return(1.0 - chi2.cdf(chisqr,ndof))

In [ ]:
search_result = lk.search_lightcurve('HD 209458', mission='TESS')
print(search_result)

In [ ]:
lc = search_result[1].download().remove_nans()

time_entire = np.array(lc.time.to_value('jd'))
flux_entire = np.array(lc.flux.to_value())

bjd_offset = 2457000.0
time_entire = time_entire - bjd_offset

In [ ]:
tc_ref = 2455420.84456 - bjd_offset # from Kokori et al. (2023), https://ui.adsabs.harvard.edu/abs/2023ApJS..265....4K/abstract
period_ref = 3.52474955             # from Kokori et al. (2023)
tra_dur_ref = 3.072/24.             # from Albrecht et al. (2012)

time_entire_folded, orbit_number = fold_time(time_entire, tc_ref, period_ref)
in_transit = np.abs(time_entire_folded) < (2.0 * tra_dur_ref)
flux_entire = flux_entire / np.median(flux_entire[~in_transit]) # normalize light curve

fig, (ax1, ax2, ax3) = plt.subplots(3,1,figsize=(20,10))

ax1.plot(time_entire, flux_entire, '.', alpha=0.3)
ax1.plot(time_entire[in_transit], flux_entire[in_transit], 'r.')
ax1.set_xlabel('Time (BTJD)')
ax1.set_ylabel('Flux (e/s)')
ax1.set_title('HD 209458 entire light curve')

ax2.plot(time_entire, orbit_number, '.')
ax2.plot(time_entire[in_transit], orbit_number[in_transit], 'r.')
ax2.set_xlabel('Time (BTJD)')
ax2.set_ylabel('Orbit number')
ax2.set_title('Orbit number of each data point')

ax3.plot(time_entire_folded, flux_entire, '.')
ax3.plot(time_entire_folded[in_transit], flux_entire[in_transit], 'r.')
ax3.set_xlabel('Folded Time (days)')
ax3.set_ylabel('Flux (e/s)')
ax3.set_title('HD 209458 entire light curve vs. folded time')

plt.tight_layout()

In [ ]:
orbit = np.unique(orbit_number)
n_transits = len(orbit) # number of transits that were observed

# Initialize some arrays that will be filled with measurements of each transit:
tc =     np.zeros(n_transits)
tc_unc = np.zeros(n_transits)
chisqr = np.zeros(n_transits)
ndof   = np.zeros(n_transits)
chisqr_prob = np.zeros(n_transits)

fig, axes = plt.subplots(n_transits, 2, figsize=(20,40))

i = 0
for o in orbit: # Analyze each transit one at a time
    
    this_transit = (orbit_number == o) & (in_transit)

    t = time_entire[this_transit]
    t_fold = time_entire_folded[this_transit]
    f = flux_entire[this_transit]
    
    out_of_transit = np.abs(t_fold) > tra_dur_ref
    f_unc = np.std(f[out_of_transit]) + 0.0*t
    
    params = lmfit.Parameters()
    params.add('period', value=period_ref, vary=False)
    params.add('tc', value=tc_ref + o*period_ref, vary=True)
    params.add('k', value=0.121, vary=True, min=0.0, max=1.0)
    params.add('a', value=8.87, vary=True, min=0.0, max=10.0)
    params.add('b', value=0.5, vary=True, min=0.0, max=1.0)
    params.add('u', value=0.6, vary=True, min=0.0, max=1.0)
    params.add('flux_oot', value=1.0, vary=True, min=0.5, max=2)
    params.add('dflux_dt', value=0.0, vary=True, min=-0.05, max=0.05)

    f_guess = tlc(params, t)
    
    axes[i,0].errorbar(t, f, f_unc, fmt='bo')
    axes[i,0].errorbar(t[out_of_transit], f[out_of_transit], f_unc[out_of_transit], fmt='ko')
    axes[i,0].plot(t, f_guess, 'g-')
    axes[i,0].set_title('Orbit number ' + str(o))
    axes[i,0].set_xlabel('Time (BJTD)')
    axes[i,0].set_ylabel('Flux')
    
    minimizer = lmfit.Minimizer(tlc_residuals, params, fcn_args=(t, f, f_unc))                    
    output = minimizer.minimize()
    params_fit = output.params
    f_calc = tlc(params_fit, t)
    axes[i,0].plot(t, f_calc, 'r-')

    axes[i,1].plot(t, (f-f_calc)/f_unc, 'o')
    axes[i,1].set_title('Orbit number ' + str(o))
    axes[i,1].set_xlabel('Time (BTJD)')
    axes[i,1].set_ylabel('Residuals')
    
    # Save the results in the output arrays that we initialized earlier
    
    chisqr[i] = output.chisqr
    ndof[i]  = output.nfree
    chisqr_prob[i] = chisqr_pval(chisqr[i], ndof[i])
    tc[i]    = params_fit['tc'].value
    tc_unc[i] = params_fit['tc'].stderr
    
    i = i+1
    
plt.tight_layout()
# plt.savefig('all_transits.pdf')

In [ ]:
print(orbit)
print(tc)
print(tc_unc)
print(chisqr)
print(ndof)
print(chisqr_prob)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1)

ax1.errorbar(orbit, tc, yerr=tc_unc, fmt='o', markersize=12)
ax1.set_xlabel('Orbit number')
ax1.set_ylabel('Transit time (BTJD)')

coeff, covar = np.polyfit(orbit, tc, 1, w=1./tc_unc, cov=True)
tc_calc = np.polyval(coeff, orbit)
ax1.plot(orbit, tc_calc, 'k-')

ax2.errorbar(orbit, 24.*60.*(tc - tc_calc), 24.*60.*tc_unc, fmt='o', markersize=12)
ax2.axhline(0, color='black')
ax2.set_xlabel('Orbit number')
ax2.set_ylabel('Transit time deviation (min)')
print(coeff)
print(np.sqrt(np.diag(covar)))


In [ ]:
x = np.array([1,2,4,4,4,4,4,4,4,43,3,3,2,1])
print(x)
print(np.unique(x))